[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_01/NB_bishop_ch01_polynomial_fitting_finance.ipynb)

# Chapter 1: The Deep Learning Revolution — Business Applications

**Bishop & Bishop (2024), Chapter 1**

This notebook implements the financial applications from the lecture slides:
- Polynomial curve fitting (Bishop tutorial example)
- S&P 500 return prediction: Linear vs MLP
- Overfitting demonstration with financial data


## Setup


In [ ]:
!pip install yfinance torch scikit-learn matplotlib numpy pandas


## 1. Polynomial Curve Fitting (Bishop Example)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
N_train, N_test = 10, 100
x_train = np.linspace(0, 1, N_train)
t_train = np.sin(2*np.pi*x_train) + np.random.randn(N_train)*0.3
x_test = np.linspace(0, 1, N_test)
t_test = np.sin(2*np.pi*x_test) + np.random.randn(N_test)*0.3

def design_matrix(x, M):
    return np.vander(x, M+1, increasing=True)

def ridge_fit(X, t, lam=0):
    I = np.eye(X.shape[1])
    return np.linalg.solve(X.T @ X + lam*I, X.T @ t)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, M in zip(axes.flat, [0, 1, 3, 9]):
    w = ridge_fit(design_matrix(x_train, M), t_train)
    y = design_matrix(x_test, M) @ w
    ax.scatter(x_train, t_train, c='blue', s=40, zorder=5)
    ax.plot(x_test, np.sin(2*np.pi*x_test), 'g--', alpha=0.5)
    ax.plot(x_test, y, 'r-', lw=2)
    rmse = np.sqrt(np.mean((y - t_test)**2))
    ax.set_title(f'M={M}, RMSE={rmse:.3f}')
plt.tight_layout()
plt.show()


## 2. S&P 500 Return Prediction: Linear vs MLP


In [ ]:
import yfinance as yf
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Download data
sp = yf.download('^GSPC', start='2015-01-01', end='2024-12-31', progress=False)
ret = sp['Close'].pct_change().dropna().values.flatten()

# Create lagged features
def make_features(ret, lags=5):
    X = np.column_stack([ret[lags-i-1:len(ret)-i-1] for i in range(lags)])
    y = ret[lags:]
    return X, y

X, y = make_features(ret)
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# Linear regression
lr = LinearRegression().fit(X_train, y_train)
rmse_lr = np.sqrt(mean_squared_error(y_test, lr.predict(X_test)))

# MLP in PyTorch
X_tr = torch.FloatTensor(X_train)
y_tr = torch.FloatTensor(y_train).unsqueeze(1)
X_te = torch.FloatTensor(X_test)

model = nn.Sequential(nn.Linear(5, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(200):
    pred = model(X_tr)
    loss = loss_fn(pred, y_tr)
    opt.zero_grad()
    loss.backward()
    opt.step()

with torch.no_grad():
    pred_test = model(X_te).numpy().flatten()
rmse_mlp = np.sqrt(mean_squared_error(y_test, pred_test))

print(f'Baseline (predict mean): RMSE = {y_test.std():.6f}')
print(f'Linear Regression:       RMSE = {rmse_lr:.6f}')
print(f'MLP (2 hidden layers):   RMSE = {rmse_mlp:.6f}')
print(f'\nR² improvement MLP vs baseline: {1 - rmse_mlp**2/y_test.var():.4%}')


## 3. Direction Accuracy and Sharpe Ratio


In [ ]:
# Direction accuracy
dir_lr = np.mean(np.sign(lr.predict(X_test)) == np.sign(y_test))
dir_mlp = np.mean(np.sign(pred_test) == np.sign(y_test))

# Simple long/short Sharpe
def sharpe(pred, actual):
    signals = np.sign(pred)
    returns = signals * actual
    return returns.mean() / returns.std() * np.sqrt(252)

print(f'Direction accuracy - Linear: {dir_lr:.1%}, MLP: {dir_mlp:.1%}')
print(f'Annualized Sharpe  - Linear: {sharpe(lr.predict(X_test), y_test):.2f}, MLP: {sharpe(pred_test, y_test):.2f}')
